In [1]:
import os
from tqdm import tqdm

In [2]:
import pandas as pd

In [3]:
from PIL import Image

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [5]:
import torchvision.models as models
import torchvision.transforms as transform

In [6]:
class ReidCatsDatasetTest(Dataset):
    def __init__(self, csv_file, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        path_image = self.annotations.iloc[index, 2]
        image = Image.open(path_image).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image

In [7]:
path_csv_test = '../data/processed/test/df_test.csv'
transformaciones = transform.Compose([
    transform.ToTensor(), 
    transform.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
    ])

In [8]:
df_test = ReidCatsDatasetTest(
    csv_file = path_csv_test,
    transform = transformaciones
    )

In [1]:
class ReidCatsDatasetTrain(Dataset):
    def __init__(self, csv_file, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, index):
        path_anchor = self.annotations.iloc[index, 2]
        path_positive = self.annotations.iloc[index, 3]
        path_negative = self.annotations.iloc[index, 4]

        anchor = Image.open(path_anchor).convert('RGB')
        positive = Image.open(path_positive).convert('RGB')
        negative = Image.open(path_negative).convert('RGB')

        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative

NameError: name 'Dataset' is not defined

In [10]:
path_csv_train = '../data/processed/train/df_train_tripletas.csv'

df_train = ReidCatsDatasetTrain(
    csv_file = path_csv_train,
    transform = transformaciones
    )

dataloader_train = DataLoader(dataset=df_train, batch_size=16, shuffle=True, num_workers=4)

In [12]:
model_1 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_1.fc = nn.Linear(model_1.fc.in_features, 512)

model_1 = model_1.to(device)

triplet_loss_fn = nn.TripletMarginLoss(margin=0.5, p=2.0)
optimizer_model_1 = optim.Adam(model_1.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\castr/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:25<00:00, 1.84MB/s]


In [13]:
def get_embeddings(x):
    return F.normalize(model_1(x), p=2, dim=1)

def train_resnet50(epochs = 5, save_path="mejor_modelo.pth"):
    loss_per_epoch = []
    best_loss = float('inf')
    
    for epoch in tqdm(range(epochs)):
        model_1.train()
        train_loss = 0.0

        for anchor, positive, negative in dataloader_train:
            anchor = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)

            batch = torch.cat([anchor, positive, negative])
            embeds = get_embeddings(batch)
            a_emb, p_emb, n_emb = embeds.chunk(3)
            loss = triplet_loss_fn(a_emb, p_emb, n_emb)

            optimizer_model_1.zero_grad()
            loss.backward()
            optimizer_model_1.step()

            train_loss += loss.item()

        avg_loss = train_loss / len(dataloader_train)   
        loss_per_epoch.append(avg_loss)
        print(f"Epoca {epoch + 1}/{epochs} - loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model_1.state_dict(), save_path)

    return train_loss, loss_per_epoch

In [ ]:
resulto_model_1 = train_resnet50()

  0%|          | 0/5 [00:00<?, ?it/s]